In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
# !wget $data -O module3_hw.csv

df = pd.read_csv('module3_hw.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


# data preparation

In [2]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
# converted is our target variable.

df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
numerical_columns = df.dtypes[df.dtypes != 'object'].index.to_list()
numerical_columns.remove('converted') # converted target varible so eta bad.

categorical_columns = df.dtypes[df.dtypes == 'object'].index.to_list()

df[numerical_columns] = df[numerical_columns].fillna(0)
df[categorical_columns] = df[categorical_columns].fillna('NA')

In [5]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# q-1

In [6]:
df.industry.mode()

0    technology
Name: industry, dtype: object

# q-2

In [7]:
df[numerical_columns].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [8]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop = True)
df_val = df_val.reset_index(drop = True)
df_test = df_test.reset_index(drop = True)

Y_train = df_train.converted.values
Y_val = df_val.converted.values
Y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

# q-3

In [9]:
from sklearn.metrics import mutual_info_score

def my_mutual_info(series):
    return mutual_info_score(series, Y_train)

mi_score = df_train[categorical_columns].apply(my_mutual_info).sort_values(ascending = False)

round(mi_score, 2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

# q-4

### one hot encoding

In [10]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse = False)

df_train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient = 'records')
X_train = dv.fit_transform(df_train_dict)

df_val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient = 'records')
X_val = dv.transform(df_val_dict)

### logistic regression 

In [11]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, Y_train)

Y_pred = model.predict_proba(X_val)[:, 1]
decision_converted = (Y_pred >= 0.5)
accuracy = (decision_converted == Y_val).mean()

round(accuracy, 2)


np.float64(0.64)